# Capstone Model 4 Analysis

This notebook contains the exploratory data analysis, feature engineering, clustering, and evaluation for Model 4.

## 1. Environment Setup & Data Import
Import required packages and load raw datasets from `../data/`.

In [1]:
import pandas as pd
import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

customers = pd.read_csv("../data/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/product_category_name_translation.csv")

# Dedupe geolocation (multiple lat/lng rows per zip prefix)
geolocation = geolocation.groupby('geolocation_zip_code_prefix').first().reset_index()

# Aggregate payments to order-level (avoid fan-out from installment rows)
payments_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode().iat[0] if not x.mode().empty else np.nan),
    n_payment_methods=('payment_type', 'nunique')
).reset_index()

# Aggregate reviews to order-level (avoid fan-out from resubmitted reviews)
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])
reviews_agg = (reviews.sort_values('review_answer_timestamp')
            .groupby('order_id').last().reset_index()
            [['order_id', 'review_score', 'review_comment_message']])

# Merge
df = orders.merge(order_items, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(reviews_agg, on='order_id', how='left')
df = df.merge(category_translation, on='product_category_name', how='left')
df = df.merge(
    geolocation,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

df_raw = df.copy()

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

print("\nRandom Sample:")
display(df.sample(5, random_state=SEED))

Shape: 113425 rows, 41 columns

First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_type,n_payment_methods,review_score,review_comment_message,product_category_name_english,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,voucher,2.0,4.0,"Não testei o produto ainda, mas ele veio corre...",housewares,3149.0,-23.574809,-46.587471,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,boleto,1.0,4.0,Muito bom o produto.,perfumery,47813.0,-12.169860,-44.988369,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,credit_card,1.0,5.0,None,auto,75265.0,-16.746337,-48.514624,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,credit_card,1.0,5.0,O produto foi exatamente o que eu esperava e e...,pet_shop,59296.0,-5.767733,-35.275467,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,credit_card,1.0,5.0,None,stationery,9195.0,-23.675037,-46.524784,santo andre,SP



Last 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_type,n_payment_methods,review_score,review_comment_message,product_category_name_english,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
113420,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,1.0,f1d4ce8c6dd66c47bbaa8c6781c2a923,...,credit_card,1.0,4.0,So uma peça que veio rachado mas tudo bem rs,baby,11722.0,-24.001467,-46.446355,praia grande,SP
113421,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,1.0,b80910977a37536adeddd63663f916ad,...,credit_card,1.0,5.0,Foi entregue antes do prazo.,home_appliances_2,45920.0,-17.891522,-39.370942,nova vicosa,BA
113422,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,credit_card,1.0,2.0,Foi entregue somente 1. Quero saber do outro p...,computers_accessories,28685.0,-22.555985,-42.690761,japuiba,RJ
113423,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,2.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,credit_card,1.0,2.0,Foi entregue somente 1. Quero saber do outro p...,computers_accessories,28685.0,-22.555985,-42.690761,japuiba,RJ
113424,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-03-08 20:57:30,2018-03-09 11:20:28,2018-03-09 22:11:59,2018-03-16 13:08:30,2018-04-03 00:00:00,1.0,006619bbed68b000c8ba3f8725d5409e,...,debit_card,1.0,5.0,None,health_beauty,83750.0,-25.775722,-49.723981,lapa,PR



Random Sample:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_type,n_payment_methods,review_score,review_comment_message,product_category_name_english,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
72065,04c3bddd55df58e83123e3b174855489,cf45e9bc7544ba97a0db03c7c7bf63c9,delivered,2018-01-26 21:34:07,2018-01-26 21:58:36,2018-02-14 20:53:13,2018-03-01 19:07:36,2018-03-02 00:00:00,1.0,f4f4debbcfcafe6858d1e37a1f6e436e,...,credit_card,1.0,1.0,Holá!\r\nNão recebi o produto é nem um telefon...,office_furniture,9961.0,-23.716539,-46.601640,diadema,SP
651,118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08 19:06:05,2018-03-09 19:08:26,2018-03-13 21:24:28,2018-04-11 12:53:50,2018-04-04 00:00:00,2.0,2c4930c4b284c7b99db2a4c52071a45e,...,credit_card,1.0,1.0,Eu pedi 6 trios de pendentes e vcs só M entreg...,furniture_decor,44900.0,-11.301811,-41.850664,irece,BA
69732,a29c25c2c9c3100daf48a25880312c23,00f394e6fc446865ac4097b6db69ef4a,delivered,2018-08-01 02:02:28,2018-08-01 02:15:06,2018-08-01 15:35:00,2018-08-08 18:58:43,2018-08-27 00:00:00,3.0,9ac1378f05cd222b3fb34a3cccc626c7,...,credit_card,1.0,1.0,Meu produto veio faltando com 5 quantidades e ...,health_beauty,26574.0,-22.778269,-43.400627,mesquita,RJ
44231,8421e7437b4c46dcf0e4895873ed8726,491dfc36c496e8880a5621b8ce98a2b6,delivered,2018-01-10 23:58:35,2018-01-11 18:27:40,2018-01-12 20:29:14,2018-01-18 00:32:21,2018-02-05 00:00:00,1.0,530fa1d000866012c51ce412598ef24c,...,boleto,1.0,5.0,Gostei muito da agilidade na entrega é na qual...,fashion_bags_accessories,74356.0,-16.788713,-49.378070,goiania,GO
107983,11ac39053e8c6e289d39a845c1a9c3b0,31f74614d63dae6a838d639204a889dc,delivered,2017-08-21 20:40:53,2017-08-21 20:55:22,2017-08-23 18:22:33,2017-09-06 22:39:43,2017-09-21 00:00:00,1.0,51d646c5c93e0f1de543528d0e24eadc,...,credit_card,1.0,5.0,None,health_beauty,52081.0,-8.005934,-34.915880,recife,PE


## 2. Model Training & Evaluation
Train clustering/predictive models, save plots to `../visuals/`, and export the best model to `../model/best_model.pkl`.

In [ ]:
# Placeholder: Model building & export
# import pickle
# with open('../model/best_model.pkl', 'wb') as f:
#     pickle.dump(best_model, f)